In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import py3Dmol

In [3]:
def find_project_root(starting_directory=None):
    """
    Search upward until a directory containing 'xyz_files'
    is found.
    """
    start = (
        Path(starting_directory).resolve()
        if starting_directory
        else Path.cwd().resolve()
    )

    for directory in [start, *start.parents]:
        if (directory / "xyz_files").is_dir():
            return directory

    raise FileNotFoundError(
        "Could not find a folder containing 'xyz_files'. "
        "Open the GSCDB Benchmarking project folder in VS Code."
    )


PROJECT_ROOT = find_project_root()
XYZ_DIR = PROJECT_ROOT / "xyz_files"

minimum_path = XYZ_DIR / "BH28_BHPERI_4_min.xyz"
ts_path = XYZ_DIR / "BH28_BHPERI_4_ts.xyz"

print("Project root:", PROJECT_ROOT)
print("XYZ directory:", XYZ_DIR)

print("\nFile checks:")
print("Minimum exists:", minimum_path.exists())
print("Transition state exists:", ts_path.exists())

Project root: C:\Users\91988\GSCDB Benchmarking
XYZ directory: C:\Users\91988\GSCDB Benchmarking\xyz_files

File checks:
Minimum exists: True
Transition state exists: True


In [4]:
def read_xyz_file(file_path):
    """
    Read and validate an XYZ molecular structure file.

    Returns
    -------
    xyz_text : str
        Full file contents for py3Dmol.

    atoms : list[dict]
        Element symbols, atom numbers and XYZ coordinates.

    comment : str
        Metadata stored on the second line.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"XYZ file not found: {file_path}"
        )

    xyz_text = file_path.read_text(encoding="utf-8")
    lines = xyz_text.splitlines()

    if len(lines) < 3:
        raise ValueError(
            f"{file_path.name} is not a valid XYZ file."
        )

    try:
        number_of_atoms = int(lines[0].strip())
    except ValueError as error:
        raise ValueError(
            f"The first line of {file_path.name} "
            "does not contain an atom count."
        ) from error

    comment = lines[1].strip()
    coordinate_lines = lines[2 : 2 + number_of_atoms]

    if len(coordinate_lines) != number_of_atoms:
        raise ValueError(
            f"{file_path.name}: expected {number_of_atoms} atoms "
            f"but found {len(coordinate_lines)} coordinate lines."
        )

    atoms = []

    for atom_number, line in enumerate(
        coordinate_lines,
        start=1
    ):
        parts = line.split()

        if len(parts) < 4:
            raise ValueError(
                f"Invalid coordinate line: {line}"
            )

        element = parts[0]
        x, y, z = map(float, parts[1:4])

        atoms.append(
            {
                "atom_number": atom_number,
                "element": element,
                "x": x,
                "y": y,
                "z": z,
            }
        )

    return xyz_text, atoms, comment

In [5]:
minimum_xyz, minimum_atoms, minimum_comment = read_xyz_file(
    minimum_path
)

ts_xyz, ts_atoms, ts_comment = read_xyz_file(
    ts_path
)

print("Minimum atoms:", len(minimum_atoms))
print("Transition-state atoms:", len(ts_atoms))

print("\nMinimum metadata:")
print(minimum_comment)

print("\nTransition-state metadata:")
print(ts_comment)

Minimum atoms: 11
Transition-state atoms: 11

Minimum metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, mem_total=3750, num_basis=513, num_pairs=129002, num_threads=1

Transition-state metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, mem_total=3750, num_basis=513, num_pairs=129149, num_threads=1


In [6]:
def molecular_formula(atoms):
    counts = Counter(
        atom["element"] for atom in atoms
    )

    order = ["C", "H", "N", "O", "F", "P", "S", "Cl", "Br"]
    formula = []

    for element in order:
        if element in counts:
            number = counts.pop(element)

            formula.append(
                element if number == 1 else f"{element}{number}"
            )

    for element in sorted(counts):
        number = counts[element]

        formula.append(
            element if number == 1 else f"{element}{number}"
        )

    return "".join(formula)


print(
    "Minimum formula:",
    molecular_formula(minimum_atoms)
)

print(
    "Transition-state formula:",
    molecular_formula(ts_atoms)
)

Minimum formula: C5H6
Transition-state formula: C5H6


In [ ]:
def molecular_formula(atoms):
    counts = Counter(
        atom["element"] for atom in atoms
    )

    order = ["C", "H", "N", "O", "F", "P", "S", "Cl", "Br"]
    formula = []

    for element in order:
        if element in counts:
            number = counts.pop(element)

            formula.append(
                element if number == 1 else f"{element}{number}"
            )

    for element in sorted(counts):
        number = counts[element]

        formula.append(
            element if number == 1 else f"{element}{number}"
        )

    return "".join(formula)


print(
    "Minimum formula:",
    molecular_formula(minimum_atoms)
)

print(
    "Transition-state formula:",
    molecular_formula(ts_atoms)
)

Minimum formula: C5H6
Transition-state formula: C5H6


In [9]:
def visualise_xyz(
    xyz_text,
    width=650,
    height=480,
    background="white"
):
    viewer = py3Dmol.view(
        width=width,
        height=height
    )

    viewer.addModel(
        xyz_text,
        "xyz"
    )

    viewer.setStyle(
        {},
        {
            "stick": {
                "radius": 0.15
            },
            "sphere": {
                "scale": 0.28
            }
        }
    )

    viewer.setBackgroundColor(background)
    viewer.zoomTo()

    return viewer

In [10]:
minimum_view = visualise_xyz(
    minimum_xyz
)

minimum_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
ts_view = visualise_xyz(
    ts_xyz
)

ts_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
comparison_view = py3Dmol.view(
    width=1100,
    height=500,
    viewergrid=(1, 2),
    linked=False
)

# Minimum: left panel
comparison_view.addModel(
    minimum_xyz,
    "xyz",
    viewer=(0, 0)
)

comparison_view.setStyle(
    {},
    {
        "stick": {"radius": 0.15},
        "sphere": {"scale": 0.28}
    },
    viewer=(0, 0)
)

comparison_view.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

comparison_view.zoomTo(
    viewer=(0, 0)
)

# Transition state: right panel
comparison_view.addModel(
    ts_xyz,
    "xyz",
    viewer=(0, 1)
)

comparison_view.setStyle(
    {},
    {
        "stick": {"radius": 0.15},
        "sphere": {"scale": 0.28}
    },
    viewer=(0, 1)
)

comparison_view.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

comparison_view.zoomTo(
    viewer=(0, 1)
)

comparison_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
def print_atom_table(atoms, structure_name):
    print(f"\n{structure_name}")
    print("-" * 64)

    print(
        f"{'Atom':>6}"
        f"{'Element':>10}"
        f"{'X / Å':>14}"
        f"{'Y / Å':>14}"
        f"{'Z / Å':>14}"
    )

    print("-" * 64)

    for atom in atoms:
        print(
            f"{atom['atom_number']:>6}"
            f"{atom['element']:>10}"
            f"{atom['x']:>14.6f}"
            f"{atom['y']:>14.6f}"
            f"{atom['z']:>14.6f}"
        )


print_atom_table(
    minimum_atoms,
    "BH28_BHPERI_4_min"
)

print_atom_table(
    ts_atoms,
    "BH28_BHPERI_4_ts"
)


BH28_BHPERI_4_min
----------------------------------------------------------------
  Atom   Element         X / Å         Y / Å         Z / Å
----------------------------------------------------------------
     1         H     -0.874940      0.000000      1.872729
     2         C      0.000000      0.000000      1.212864
     3         C      0.000000      1.176358      0.280855
     4         C      0.000000      0.732369     -0.987565
     5         C      0.000000     -0.732369     -0.987565
     6         C      0.000000     -1.176358      0.280855
     7         H      0.874940      0.000000      1.872729
     8         H      0.000000     -2.204926      0.606181
     9         H      0.000000      2.204926      0.606181
    10         H      0.000000      1.344827     -1.877240
    11         H      0.000000     -1.344827     -1.877240

BH28_BHPERI_4_ts
----------------------------------------------------------------
  Atom   Element         X / Å         Y / Å         Z / Å
-

In [14]:
def atom_position(atoms, atom_number):
    """
    Return the Cartesian coordinates of a one-based atom number.
    """
    if atom_number < 1 or atom_number > len(atoms):
        raise IndexError(
            f"Atom number must be between 1 and {len(atoms)}."
        )

    atom = atoms[atom_number - 1]

    return np.array(
        [
            atom["x"],
            atom["y"],
            atom["z"],
        ],
        dtype=float
    )


def distance_between_atoms(
    atoms,
    atom_number_1,
    atom_number_2
):
    position_1 = atom_position(
        atoms,
        atom_number_1
    )

    position_2 = atom_position(
        atoms,
        atom_number_2
    )

    return float(
        np.linalg.norm(position_1 - position_2)
    )

In [15]:
minimum_donor_h = distance_between_atoms(
    minimum_atoms,
    2,
    1
)

minimum_acceptor_h = distance_between_atoms(
    minimum_atoms,
    6,
    1
)

print(
    f"Minimum donor C2-H1 distance: "
    f"{minimum_donor_h:.3f} Å"
)

print(
    f"Minimum H1···acceptor C6 distance: "
    f"{minimum_acceptor_h:.3f} Å"
)

Minimum donor C2-H1 distance: 1.096 Å
Minimum H1···acceptor C6 distance: 2.164 Å


In [16]:
minimum_other_direction = distance_between_atoms(
    minimum_atoms,
    3,
    1
)

print(
    f"Minimum H1···C3 distance: "
    f"{minimum_other_direction:.3f} Å"
)

Minimum H1···C3 distance: 2.164 Å


In [17]:
ts_h_c1 = distance_between_atoms(
    ts_atoms,
    6,
    1
)

ts_h_c2 = distance_between_atoms(
    ts_atoms,
    6,
    2
)

print(
    f"TS H6···C1 distance: "
    f"{ts_h_c1:.3f} Å"
)

print(
    f"TS H6···C2 distance: "
    f"{ts_h_c2:.3f} Å"
)

TS H6···C1 distance: 1.309 Å
TS H6···C2 distance: 1.309 Å


In [18]:
minimum_ring_bonds = [
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 2),
]

print("Minimum ring distances:\n")

for atom_1, atom_2 in minimum_ring_bonds:
    distance = distance_between_atoms(
        minimum_atoms,
        atom_1,
        atom_2
    )

    print(
        f"C{atom_1}-C{atom_2}: "
        f"{distance:.3f} Å"
    )

Minimum ring distances:

C2-C3: 1.501 Å
C3-C4: 1.344 Å
C4-C5: 1.465 Å
C5-C6: 1.344 Å
C6-C2: 1.501 Å


In [19]:
ts_ring_bonds = [
    (1, 3),
    (3, 5),
    (5, 4),
    (4, 2),
    (2, 1),
]

print("Transition-state ring distances:\n")

for atom_1, atom_2 in ts_ring_bonds:
    distance = distance_between_atoms(
        ts_atoms,
        atom_1,
        atom_2
    )

    print(
        f"C{atom_1}-C{atom_2}: "
        f"{distance:.3f} Å"
    )

Transition-state ring distances:

C1-C3: 1.405 Å
C3-C5: 1.397 Å
C5-C4: 1.397 Å
C4-C2: 1.405 Å
C2-C1: 1.486 Å


In [20]:
def coordinate_dictionary(atoms, atom_number):
    atom = atoms[atom_number - 1]

    return {
        "x": atom["x"],
        "y": atom["y"],
        "z": atom["z"],
    }


def midpoint_dictionary(
    atoms,
    atom_number_1,
    atom_number_2
):
    point_1 = atom_position(
        atoms,
        atom_number_1
    )

    point_2 = atom_position(
        atoms,
        atom_number_2
    )

    midpoint = (
        point_1 + point_2
    ) / 2.0

    return {
        "x": float(midpoint[0]),
        "y": float(midpoint[1]),
        "z": float(midpoint[2]),
    }

In [21]:
highlight_view = py3Dmol.view(
    width=1150,
    height=520,
    viewergrid=(1, 2),
    linked=False
)

# ==================================================
# Left panel: minimum
# ==================================================

highlight_view.addModel(
    minimum_xyz,
    "xyz",
    viewer=(0, 0)
)

highlight_view.setStyle(
    {},
    {
        "stick": {"radius": 0.14},
        "sphere": {"scale": 0.27}
    },
    viewer=(0, 0)
)

# Existing donor C-H bond: C2-H1
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            minimum_atoms,
            2
        ),
        "end": coordinate_dictionary(
            minimum_atoms,
            1
        ),
        "radius": 0.055,
        "color": "blue",
        "opacity": 0.9,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 0)
)

# Possible migration direction: H1···C6
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            minimum_atoms,
            1
        ),
        "end": coordinate_dictionary(
            minimum_atoms,
            6
        ),
        "radius": 0.025,
        "color": "grey",
        "opacity": 0.55,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 0)
)

highlight_view.addLabel(
    f"C2-H1 = {minimum_donor_h:.3f} Å",
    {
        "position": midpoint_dictionary(
            minimum_atoms,
            2,
            1
        ),
        "backgroundColor": "white",
        "fontColor": "blue",
        "fontSize": 14,
        "showBackground": True,
    },
    viewer=(0, 0)
)

highlight_view.addLabel(
    f"H1···C6 = {minimum_acceptor_h:.3f} Å",
    {
        "position": midpoint_dictionary(
            minimum_atoms,
            1,
            6
        ),
        "backgroundColor": "white",
        "fontColor": "grey",
        "fontSize": 14,
        "showBackground": True,
    },
    viewer=(0, 0)
)

highlight_view.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

highlight_view.zoomTo(
    viewer=(0, 0)
)

# ==================================================
# Right panel: transition state
# ==================================================

highlight_view.addModel(
    ts_xyz,
    "xyz",
    viewer=(0, 1)
)

highlight_view.setStyle(
    {},
    {
        "stick": {"radius": 0.14},
        "sphere": {"scale": 0.27}
    },
    viewer=(0, 1)
)

# TS H···C interaction 1
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            ts_atoms,
            6
        ),
        "end": coordinate_dictionary(
            ts_atoms,
            1
        ),
        "radius": 0.05,
        "color": "red",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 1)
)

# TS H···C interaction 2
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            ts_atoms,
            6
        ),
        "end": coordinate_dictionary(
            ts_atoms,
            2
        ),
        "radius": 0.05,
        "color": "red",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 1)
)

highlight_view.addLabel(
    f"H6···C1 = {ts_h_c1:.3f} Å",
    {
        "position": midpoint_dictionary(
            ts_atoms,
            6,
            1
        ),
        "backgroundColor": "white",
        "fontColor": "red",
        "fontSize": 14,
        "showBackground": True,
    },
    viewer=(0, 1)
)

highlight_view.addLabel(
    f"H6···C2 = {ts_h_c2:.3f} Å",
    {
        "position": midpoint_dictionary(
            ts_atoms,
            6,
            2
        ),
        "backgroundColor": "white",
        "fontColor": "red",
        "fontSize": 14,
        "showBackground": True,
    },
    viewer=(0, 1)
)

highlight_view.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

highlight_view.zoomTo(
    viewer=(0, 1)
)

highlight_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [24]:
def coordinate_dictionary(atoms, atom_number):
    """
    Return the XYZ coordinates of an atom in the dictionary
    format required by py3Dmol.

    atom_number uses one-based XYZ numbering.
    """
    atom = atoms[atom_number - 1]

    return {
        "x": float(atom["x"]),
        "y": float(atom["y"]),
        "z": float(atom["z"]),
    }


def midpoint_dictionary(
    atoms,
    atom_number_1,
    atom_number_2,
    offset=(0.0, 0.0, 0.0),
):
    """
    Return the midpoint between two atoms.

    The optional offset can move a distance label away from
    the molecular structure to reduce overlap.
    """
    point_1 = atom_position(
        atoms,
        atom_number_1
    )

    point_2 = atom_position(
        atoms,
        atom_number_2
    )

    midpoint = (point_1 + point_2) / 2.0

    dx, dy, dz = offset
    midpoint = midpoint + np.array(
        [dx, dy, dz],
        dtype=float
    )

    return {
        "x": float(midpoint[0]),
        "y": float(midpoint[1]),
        "z": float(midpoint[2]),
    }

In [25]:
def xyz_atom_label(atom):
    """
    Create an atom label using the element and original
    XYZ row number.

    Examples
    --------
    Atom 1, element H -> H1
    Atom 2, element C -> C2
    Atom 11, element H -> H11
    """
    return f"{atom['element']}{atom['atom_number']}"

In [26]:
def atom_label_position(
    atoms,
    atom_number,
    carbon_offset=0.28,
    hydrogen_offset=0.38,
):
    """
    Move an atom label slightly away from the molecular centre
    so that the text is not hidden by the atom sphere.
    """
    coordinates = np.array(
        [
            [atom["x"], atom["y"], atom["z"]]
            for atom in atoms
        ],
        dtype=float
    )

    molecular_centre = coordinates.mean(axis=0)

    atom = atoms[atom_number - 1]

    atom_coordinates = np.array(
        [
            atom["x"],
            atom["y"],
            atom["z"],
        ],
        dtype=float
    )

    outward_vector = atom_coordinates - molecular_centre
    vector_length = np.linalg.norm(outward_vector)

    # Prevent division by zero if an atom is very close
    # to the calculated molecular centre.
    if vector_length < 1.0e-12:
        outward_vector = np.array(
            [0.0, 0.0, 1.0],
            dtype=float
        )
        vector_length = 1.0

    outward_unit_vector = (
        outward_vector / vector_length
    )

    if atom["element"] == "H":
        offset_distance = hydrogen_offset
    else:
        offset_distance = carbon_offset

    label_coordinates = (
        atom_coordinates
        + offset_distance * outward_unit_vector
    )

    return {
        "x": float(label_coordinates[0]),
        "y": float(label_coordinates[1]),
        "z": float(label_coordinates[2]),
    }

In [27]:
def add_all_atom_labels(
    viewer,
    atoms,
    panel,
    font_size=11,
):
    """
    Add the XYZ atom number to every carbon and hydrogen atom.

    Carbon labels are shown in black.
    Hydrogen labels are shown in blue.
    """
    for atom in atoms:
        element = atom["element"]

        # This dataset contains C and H, but this condition
        # prevents unexpected elements from being labelled.
        if element not in {"C", "H"}:
            continue

        if element == "C":
            font_colour = "black"
        else:
            font_colour = "blue"

        viewer.addLabel(
            xyz_atom_label(atom),
            {
                "position": atom_label_position(
                    atoms,
                    atom["atom_number"]
                ),
                "fontColor": font_colour,
                "backgroundColor": "white",
                "backgroundOpacity": 0.75,
                "fontSize": font_size,
                "showBackground": True,
                "inFront": True,
            },
            viewer=panel
        )

In [28]:
def print_xyz_numbering(atoms, structure_name):
    """
    Print the labels used in the molecular visualisation.
    """
    print(f"\n{structure_name}")
    print("-" * 45)
    print(
        f"{'XYZ row':>8} "
        f"{'Element':>10} "
        f"{'Displayed label':>17}"
    )
    print("-" * 45)

    for atom in atoms:
        print(
            f"{atom['atom_number']:>8} "
            f"{atom['element']:>10} "
            f"{xyz_atom_label(atom):>17}"
        )


print_xyz_numbering(
    minimum_atoms,
    "BH28_BHPERI_4_min"
)

print_xyz_numbering(
    ts_atoms,
    "BH28_BHPERI_4_ts"
)


BH28_BHPERI_4_min
---------------------------------------------
 XYZ row    Element   Displayed label
---------------------------------------------
       1          H                H1
       2          C                C2
       3          C                C3
       4          C                C4
       5          C                C5
       6          C                C6
       7          H                H7
       8          H                H8
       9          H                H9
      10          H               H10
      11          H               H11

BH28_BHPERI_4_ts
---------------------------------------------
 XYZ row    Element   Displayed label
---------------------------------------------
       1          C                C1
       2          C                C2
       3          C                C3
       4          C                C4
       5          C                C5
       6          H                H6
       7          H                H7
       8          H

In [29]:
highlight_view = py3Dmol.view(
    width=1250,
    height=600,
    viewergrid=(1, 2),
    linked=False
)

# ==========================================================
# LEFT PANEL: BH28_BHPERI_4_min
# ==========================================================

highlight_view.addModel(
    minimum_xyz,
    "xyz",
    viewer=(0, 0)
)

highlight_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.13
        },
        "sphere": {
            "scale": 0.25
        }
    },
    viewer=(0, 0)
)

# Existing donor C-H bond: C2-H1
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            minimum_atoms,
            2
        ),
        "end": coordinate_dictionary(
            minimum_atoms,
            1
        ),
        "radius": 0.055,
        "color": "blue",
        "opacity": 0.90,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 0)
)

# Possible hydrogen-migration direction: H1···C6
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            minimum_atoms,
            1
        ),
        "end": coordinate_dictionary(
            minimum_atoms,
            6
        ),
        "radius": 0.025,
        "color": "grey",
        "opacity": 0.60,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 0)
)

# Donor C-H distance label
highlight_view.addLabel(
    f"C2-H1 = {minimum_donor_h:.3f} Å",
    {
        "position": midpoint_dictionary(
            minimum_atoms,
            2,
            1,
            offset=(0.0, 0.0, 0.35)
        ),
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "blue",
        "fontSize": 13,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 0)
)

# H···acceptor-C distance label
highlight_view.addLabel(
    f"H1···C6 = {minimum_acceptor_h:.3f} Å",
    {
        "position": midpoint_dictionary(
            minimum_atoms,
            1,
            6,
            offset=(0.0, 0.0, -0.35)
        ),
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "grey",
        "fontSize": 13,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 0)
)

# Structure title
highlight_view.addLabel(
    "BH28_BHPERI_4_min",
    {
        "position": {
            "x": 0.0,
            "y": 0.0,
            "z": 3.8
        },
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 16,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 0)
)

# Add C2, C3, ..., H1, H7, etc.
add_all_atom_labels(
    viewer=highlight_view,
    atoms=minimum_atoms,
    panel=(0, 0),
    font_size=11
)

highlight_view.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

highlight_view.zoomTo(
    viewer=(0, 0)
)


# ==========================================================
# RIGHT PANEL: BH28_BHPERI_4_ts
# ==========================================================

highlight_view.addModel(
    ts_xyz,
    "xyz",
    viewer=(0, 1)
)

highlight_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.13
        },
        "sphere": {
            "scale": 0.25
        }
    },
    viewer=(0, 1)
)

# First TS C···H interaction: C1···H6
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            ts_atoms,
            1
        ),
        "end": coordinate_dictionary(
            ts_atoms,
            6
        ),
        "radius": 0.050,
        "color": "red",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 1)
)

# Second TS C···H interaction: H6···C2
highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            ts_atoms,
            6
        ),
        "end": coordinate_dictionary(
            ts_atoms,
            2
        ),
        "radius": 0.050,
        "color": "red",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 1)
)

# First TS distance label
highlight_view.addLabel(
    f"C1···H6 = {ts_h_c1:.3f} Å",
    {
        "position": midpoint_dictionary(
            ts_atoms,
            1,
            6,
            offset=(0.0, 0.0, 0.38)
        ),
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "red",
        "fontSize": 13,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 1)
)

# Second TS distance label
highlight_view.addLabel(
    f"H6···C2 = {ts_h_c2:.3f} Å",
    {
        "position": midpoint_dictionary(
            ts_atoms,
            6,
            2,
            offset=(0.0, 0.0, -0.38)
        ),
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "red",
        "fontSize": 13,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 1)
)

# Structure title
highlight_view.addLabel(
    "BH28_BHPERI_4_ts",
    {
        "position": {
            "x": 0.0,
            "y": 0.0,
            "z": 3.8
        },
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 16,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 1)
)

# Add C1, C2, ..., H6, H7, etc.
add_all_atom_labels(
    viewer=highlight_view,
    atoms=ts_atoms,
    panel=(0, 1),
    font_size=11
)

highlight_view.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

highlight_view.zoomTo(
    viewer=(0, 1)
)

# Display the final interactive visualisation
highlight_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.